<a href="https://colab.research.google.com/github/mmilannaik/bostonhousepricing/blob/main/W15S3_SQL_Subquery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the Kaggle CLI
!pip install kaggle --quiet

# 2. Upload your Kaggle API token
#    • On Kaggle: Account → Create New API Token → download kaggle.json
#    • In Colab:
from google.colab import files
files.upload()   # select your kaggle.json

# 3. Configure the CLI
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. Download & unzip the dataset
!kaggle datasets download -d equilibriumm/sleep-efficiency
!unzip -q sleep-efficiency.zip   # adjust if the zip has a folder

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/equilibriumm/sleep-efficiency
License(s): copyright-authors


In [2]:

# 1. Install pandasql
!pip install pandasql --quiet

# 2. Load libraries and your CSV into a pandas DataFrame
import pandas as pd
from pandasql import sqldf

# 3. Create a helper to run SQL against any DataFrame in your notebook
pysqldf = lambda query: sqldf(query, globals())

  Preparing metadata (setup.py) ... done


In [3]:
movies = pd.read_csv('/content/W15S2_movies.csv', sep=';',
                     quotechar='"',encoding='latin-1')

In [4]:
movies.head(2)

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000,58853106.0,Columbia Pictures,104.0


In [5]:
pysqldf('''
SELECT * FROM movies
where score =(
SELECT MAX(score) FROM movies)

''')

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shawshank Redemption,R,Drama,1994,"October 14, 1994 (United States)",9.3,2400000.0,Frank Darabont,Stephen King,Tim Robbins,United States,25000000,28817291.0,Castle Rock Entertainment,142.0


# Independent Sub query-Scalar Sub query

## Schema

In [6]:
movies.head(2)

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000,58853106.0,Columbia Pictures,104.0


 ## Q1 : Find the most profitable movie

In [7]:
pysqldf('''
SELECT  * FROM movies
where  gross - budget =
(SELECT MAX(gross-budget) FROM movies)

''')

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,Titanic,PG-13,Drama,1997,"December 19, 1997 (United States)",7.8,1100000.0,James Cameron,James Cameron,Leonardo DiCaprio,United States,200000000,2.201647e+09,Twentieth Century Fox,194.0


Here searching fast in below

In [8]:
pysqldf('''
SELECT  * FROM movies
ORDER BY (gross-budget) DESC LIMIT 1

''')

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,Titanic,PG-13,Drama,1997,"December 19, 1997 (United States)",7.8,1100000.0,James Cameron,James Cameron,Leonardo DiCaprio,United States,200000000,2.201647e+09,Twentieth Century Fox,194.0


## Q2: find count of movies whose rating > avg of movie rating

In [9]:
pysqldf('''
SELECT  COUNT(*) FROM movies
WHERE score > (SELECT AVG(score) FROM movies)

''')

,COUNT(*)
0,2077


## Q3 : Highest rated movie of 2000

In [10]:
pysqldf('''
SELECT * FROM movies
WHERE year =2000 AND score = (SELECT MAX(score) FROM movies WHERE year ="2000")

''')

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,Gladiator,R,Action,2000,"May 5, 2000 (United States)",8.5,1400000.0,Ridley Scott,David Franzoni,Russell Crowe,United States,103000000,465380802.0,Dreamworks Pictures,155.0


## Q4 : Highest rates movies amongall whose humber of votes > dataset avg votes

In [11]:
pysqldf('''
SELECT * FROM movies
WHERE score = (
              SELECT MAX(score) FROM movies
              WHERE votes > (
              SELECT AVG(votes) FROM movies))


''')

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shawshank Redemption,R,Drama,1994,"October 14, 1994 (United States)",9.3,2400000.0,Frank Darabont,Stephen King,Tim Robbins,United States,25000000,28817291.0,Castle Rock Entertainment,142.0


# Indenpendent Sub query- Row subquery( one col-multi rows)

## Schema

In [12]:
orders = pd.read_csv('/content/W15S2_orders.csv')
orders.head(2)

,order_id,user_id,r_id,amount,date,partner_id,delivery_time,delivery_rating,restaurant_rating
0,1001,1,1,550,2022-05-10,1,25,5,3.0
1,1002,1,2,415,2022-05-26,1,19,5,2.0


In [13]:
users = pd.read_csv("/content/W15S2_users.csv")
users.head(2)

,user_id,name,email,password
0,1,Nitish,nitish@gmail.com,p252h
1,2,Khushboo,khushboo@gmail.com,hxn9b


## Q1 find all users who never orders

In [14]:
pysqldf(
    '''
    SELECT * FROM users
    WHERE user_id NOT IN (SELECT DISTINCT(user_id) FROM orders)
    '''

)

,user_id,name,email,password
0,6,Anupama,anupama@gmail.com,46rdw2
1,7,Rishabh,rishabh@gmail.com,4sw123


## Q2 Find allmovies made by top3 directors(in terms of totalgross income)

In [15]:
pysqldf('''
SELECT name FROM movies WHERE director IN (SELECT director FROM movies
                                            GROUP BY director
                                            ORDER BY SUM(gross) DESC LIMIT 3)

''')

,name
0,Indiana Jones and the Raiders of the Lost Ark
1,E.T. the Extra-Terrestrial
2,The Terminator
3,Indiana Jones and the Temple of Doom
4,Romancing the Stone
5,Back to the Future
6,The Color Purple
7,Aliens
8,Empire of the Sun
9,Who Framed Roger Rabbit


## Q3: Find all movies of all actors whose filmography avg rating > 8.5(take 25000 votes as cutoff)

In [16]:
pysqldf('''
SELECT name FROM movies
WHERE star  IN(
              SELECT star FROM movies
              WHERE votes > 25000
              GROUP BY star
              HAVING  AVG(score) > 8.5 )
''')

,name
0,Johnny Stecchino
1,The Adventures of Huck Finn
2,Son of the Pink Panther
3,North
4,The War
5,Life Is Beautiful
6,The Lord of the Rings: The Fellowship of the Ring
7,Spirited Away


Jahan Pe Jo hai..wahan pe subquery


# Independent Subquery-Table Subquery(Multi Col Multi Row)

## Q1 Find mostprofitable movie of each year

In [23]:
pysqldf(
    '''
    SELECT * FROM movies
    WHERE (year,gross-budget) IN (
                                    SELECT year,MAX(gross-budget) FROM movies
                                    group by year
                                    ORDER BY MAX(gross-budget) DESC )

    '''

)

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000,5.383751e+08,Lucasfilm,124.0
1,Indiana Jones and the Raiders of the Lost Ark,PG,Action,1981,"June 12, 1981 (United States)",8.4,905000.0,Steven Spielberg,Lawrence Kasdan,Harrison Ford,United States,18000000,3.899260e+08,Paramount Pictures,115.0
2,E.T. the Extra-Terrestrial,PG,Family,1982,"June 11, 1982 (United States)",7.8,381000.0,Steven Spielberg,Melissa Mathison,Henry Thomas,United States,10500000,7.929106e+08,Universal Pictures,115.0
3,Star Wars: Episode VI - Return of the Jedi,PG,Action,1983,"May 25, 1983 (United States)",8.3,973000.0,Richard Marquand,Lawrence Kasdan,Mark Hamill,United States,32500000,4.751062e+08,Lucasfilm,131.0
4,Indiana Jones and the Temple of Doom,PG,Action,1984,"May 23, 1984 (United States)",7.5,459000.0,Steven Spielberg,Willard Huyck,Harrison Ford,United States,28000000,3.331073e+08,Paramount Pictures,118.0
5,Back to the Future,PG,Adventure,1985,"July 3, 1985 (United States)",8.5,1100000.0,Robert Zemeckis,Robert Zemeckis,Michael J. Fox,United States,19000000,3.819068e+08,Universal Pictures,116.0
6,Top Gun,PG,Action,1986,"May 16, 1986 (United States)",6.9,306000.0,Tony Scott,Jim Cash,Tom Cruise,United States,15000000,3.572882e+08,Paramount Pictures,110.0
7,Fatal Attraction,R,Drama,1987,"September 18, 1987 (United States)",6.9,79000.0,Adrian Lyne,James Dearden,Michael Douglas,United States,14000000,3.201457e+08,Paramount Pictures,119.0
8,Rain Man,R,Drama,1988,"December 16, 1988 (United States)",8.0,483000.0,Barry Levinson,Barry Morrow,Dustin Hoffman,United States,25000000,3.548254e+08,United Artists,133.0
9,Indiana Jones and the Last Crusade,PG-13,Action,1989,"May 24, 1989 (United States)",8.2,707000.0,Steven Spielberg,Jeffrey Boam,Harrison Ford,United States,48000000,4.741718e+08,Paramount Pictures,127.0


## Q2:Highest rated movie of each genre votes cutoff of 25000

In [28]:
pysqldf('''
SELECT * FROM movies
WHERE (genre,score) IN (
                        SELECT genre,MAX(score) FROM movies
                        group by genre
                        HAVING votes > 25000 )


''')

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Thing,R,Horror,1982,"June 25, 1982 (United States)",8.1,382000.0,John Carpenter,Bill Lancaster,Kurt Russell,United States,15000000,19632053.0,Universal Pictures,109.0
1,E.T. the Extra-Terrestrial,PG,Family,1982,"June 11, 1982 (United States)",7.8,381000.0,Steven Spielberg,Melissa Mathison,Henry Thomas,United States,10500000,792910554.0,Universal Pictures,115.0
2,Back to the Future,PG,Adventure,1985,"July 3, 1985 (United States)",8.5,1100000.0,Robert Zemeckis,Robert Zemeckis,Michael J. Fox,United States,19000000,381906762.0,Universal Pictures,116.0
3,Schindler's List,R,Biography,1993,"February 4, 1994 (United States)",8.9,1200000.0,Steven Spielberg,Thomas Keneally,Liam Neeson,United States,22000000,322161245.0,Universal Pictures,195.0
4,The Shawshank Redemption,R,Drama,1994,"October 14, 1994 (United States)",9.3,2400000.0,Frank Darabont,Stephen King,Tim Robbins,United States,25000000,28817291.0,Castle Rock Entertainment,142.0
5,Pulp Fiction,R,Crime,1994,"October 14, 1994 (United States)",8.9,1900000.0,Quentin Tarantino,Quentin Tarantino,John Travolta,United States,8000000,213928762.0,Miramax,154.0
6,Life Is Beautiful,PG-13,Comedy,1997,"December 20, 1997 (Italy)",8.6,642000.0,Roberto Benigni,Vincenzo Cerami,Roberto Benigni,Italy,20000000,230098753.0,Melampo Cinematografica,116.0
7,Dark City,R,Fantasy,1998,"February 27, 1998 (United States)",7.6,191000.0,Alex Proyas,Alex Proyas,Rufus Sewell,Australia,27000000,27200316.0,Mystery Clock Cinema,100.0
8,Memento,R,Mystery,2000,"May 25, 2001 (United States)",8.4,1200000.0,Christopher Nolan,Christopher Nolan,Guy Pearce,United States,9000000,40047078.0,Newmarket Capital Group,113.0
9,The Lord of the Rings: The Fellowship of the Ring,PG-13,Action,2001,"December 19, 2001 (United States)",8.8,1700000.0,Peter Jackson,J.R.R. Tolkien,Elijah Wood,New Zealand,93000000,897690072.0,New Line Cinema,178.0


## Q3:Highest grossing movies of top 5 acto/director combo wrt gtoss income

In [33]:
pysqldf(
    '''

    With top_duos AS (
            SELECT star,director,MAX(gross) FROM movies
            group by star,director
            ORDER BY SUM(gross) DESC LIMIT 5 )
    SELECT * FROM movies
    WHERE (star,director,gross) IN (SELECT * FROM top_duos)

    '''


)

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,Indiana Jones and the Last Crusade,PG-13,Action,1989,"May 24, 1989 (United States)",8.2,707000.0,Steven Spielberg,Jeffrey Boam,Harrison Ford,United States,48000000,4.741718e+08,Paramount Pictures,127.0
1,Lethal Weapon 3,R,Action,1992,"May 15, 1992 (United States)",6.7,160000.0,Richard Donner,Jeffrey Boam,Mel Gibson,United States,35000000,3.217315e+08,Warner Bros.,118.0
2,Forrest Gump,PG-13,Drama,1994,"July 6, 1994 (United States)",8.8,1900000.0,Robert Zemeckis,Winston Groom,Tom Hanks,United States,55000000,6.782261e+08,Paramount Pictures,142.0
3,The Lion King,G,Animation,1994,"June 24, 1994 (United States)",8.5,970000.0,Roger Allers,Irene Mecchi,Matthew Broderick,United States,45000000,1.083721e+09,Walt Disney Pictures,88.0
4,Titanic,PG-13,Drama,1997,"December 19, 1997 (United States)",7.8,1100000.0,James Cameron,James Cameron,Leonardo DiCaprio,United States,200000000,2.201647e+09,Twentieth Century Fox,194.0


# Correlated Subquery

## Q1: All movies that have rating higher than avg rating of  movies of same genre

In [40]:
pysqldf(
    '''
    SELECT * FROM movies m1
    WHERE score > (SELECT avg(score) AS 'genre_avg' FROM movies m2 WHERE m2.genre = m1.genre)


    '''

)

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000,46998772.0,Warner Bros.,146.0
1,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000,538375067.0,Lucasfilm,124.0
2,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000,83453539.0,Paramount Pictures,88.0
3,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000,39846344.0,Orion Pictures,98.0
4,Friday the 13th,R,Horror,1980,"May 9, 1980 (United States)",6.4,123000.0,Sean S. Cunningham,Victor Miller,Betsy Palmer,United States,550000,39754601.0,Paramount Pictures,95.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2099,Buffalo Soldiers,R,Comedy,2001,"October 31, 2002 (Germany)",6.8,24000.0,Gregor Jordan,Robert O'Connor,Joaquin Phoenix,United Kingdom,15000000,2300684.0,FilmFour,98.0
2100,The Anniversary Party,R,Comedy,2001,"June 29, 2001 (United States)",6.3,8100.0,Alan Cumming,Jennifer Jason Leigh,Alan Cumming,United States,0,4931888.0,Fine Line Features,115.0
2101,Blow Dry,R,Comedy,2001,"March 30, 2001 (United Kingdom)",6.3,8100.0,Paddy Breathnach,Simon Beaufoy,Alan Rickman,United States,0,830286.0,IMF Internationale Medien und Film GmbH & Co. ...,94.0
2102,Human Nature,R,Comedy,2001,"September 12, 2001 (France)",6.4,18000.0,Michel Gondry,Charlie Kaufman,Tim Robbins,France,0,1574660.0,Fine Line Features,96.0
